# Corpus review

Checks the output of `python -m sbs corpus ...` and records manual fixes.

- Set `SBS_DB` / `SBS_CONFIG` in the environment, or edit the variables in the next cell.
- Manual fixes are written to the override files in `config/experiment.toml` `[paths]`
  (`manual_labels`, `manual_exclude`), never to the database, so they survive re-runs.
- Clear all outputs before committing this notebook: it would otherwise contain README text.

In [ ]:
import os
import random
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "sbs").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from sbs.common import settings as settings_mod
from sbs.common.store import Store
from sbs.corpus import manual
from sbs.corpus.steps import run_step
from sbs.corpus.tokens import deserialize
from sbs.review import checks, frames

DB = os.environ.get("SBS_DB", "crawler.db")
CONFIG = os.environ.get("SBS_CONFIG", "config")
SAMPLE = 5
SEED = 0

settings = settings_mod.load(CONFIG)
paths = settings["experiment"]["paths"]
store = Store(DB)
pd.set_option("display.max_colwidth", 120)
frames.stage_counts(store)

## 1. Cleaning

In [ ]:
docs = frames.repos(store)
cleaned = docs.dropna(subset=["clean"])
rng = random.Random(SEED)
for row in cleaned.loc[rng.sample(list(cleaned.index), min(SAMPLE, len(cleaned)))].itertuples():
    display(Markdown(f"### {row.full_name} (id {row.id})"))
    print("RAW:")
    print((row.raw or "")[:800])
    print()
    print("CLEAN:")
    print(row.clean[:800])

In [ ]:
ratio = cleaned["clean"].str.len() / cleaned["raw"].str.len()
ax = ratio.clip(upper=1).plot.hist(bins=50, title="clean / raw length")
ax.set_xlabel("ratio")
plt.show()
cleaned.assign(clean_len=cleaned["clean"].str.len()).nsmallest(10, "clean_len")[["id", "full_name", "clean_len", "clean"]]

In [ ]:
residue = checks.residue(store)
residue

In [ ]:
# Inspect examples of one residue type
kind = "url"
frames.repos(store, residue.loc[kind, "example_ids"])[["id", "full_name", "clean"]]

## 2. Tokens

In [ ]:
tok = frames.query(store, "SELECT repo_id, text, n_tokens FROM tokens")
tok["n_tokens"].plot.hist(bins=50, log=True, title="tokens per document")
plt.show()
tok["n_tokens"].describe()

In [ ]:
counts = Counter(w for text in tok["text"] for s in deserialize(text) for w in s)
pd.DataFrame(counts.most_common(50), columns=["token", "count"]).T

In [ ]:
technical = ["c++", "c#", "f#", "node.js", ".net", "real-time", "scikit-learn"]
pd.Series({t: counts.get(t, 0) for t in technical}, name="count")

## 3. De-duplication

In [ ]:
frames.query(store, "SELECT status, COUNT(*) AS repos FROM dedup GROUP BY status ORDER BY repos DESC")

In [ ]:
frames.duplicate_groups(store, "exact", limit=5)

In [ ]:
frames.duplicate_groups(store, "near", limit=5)

## 4. Labels

In [ ]:
frames.label_crosstab(store)

In [ ]:
conf = frames.query(store, "SELECT domain, confidence FROM labels WHERE label_source = 'model'")
if not conf.empty:
    conf["confidence"].plot.hist(bins=40, title="classifier confidence")
    plt.axvline(settings["experiment"]["corpus"]["labels"]["min_confidence"], color="red")
    plt.show()

In [ ]:
labelled = docs.dropna(subset=["domain"])
for domain, group in labelled.groupby("domain"):
    display(Markdown(f"### {domain} ({len(group)})"))
    display(group.sample(min(SAMPLE, len(group)), random_state=SEED)[
        ["id", "full_name", "label_source", "confidence", "topics", "description"]])

In [ ]:
frames.domain_agreement(store)

In [ ]:
frames.awesome_coverage(store)

## 5. Test split

In [ ]:
summary = checks.split_summary(store, paths["test_set"])
display({k: v for k, v in summary.items() if k != "per_domain"})
summary.get("per_domain")

## 6. Manual fixes

`domain` may be any domain name or `unknown` (keeps the repository out of the labelled set).
Excluded repositories get dedup status `excluded`; another member of their duplicate group can
then be kept. The frozen test set is never redrawn.

In [ ]:
# manual.set_label(paths["manual_labels"], 123, "devops", note="topic says ml, README is CI tooling")
# manual.exclude(paths["manual_exclude"], 456, reason="list of links, not a project")
# manual.remove(paths["manual_labels"], 123)
display(pd.DataFrame(manual.read_labels(paths["manual_labels"]).items(), columns=["repo_id", "domain"]))
display(pd.DataFrame(manual.read_excluded(paths["manual_exclude"]).items(), columns=["repo_id", "reason"]))

In [ ]:
# Apply the override files: re-runs dedup and label (not split)
APPLY = False
if APPLY:
    for step in ("dedup", "label"):
        run_step(step, store, settings)
    display(frames.stage_counts(store))

In [ ]:
store.close()